# V2 Hidden State Extraction — Z-Gap
**Runtime → Change runtime type → T4 GPU** 선택 후 실행

모델 2개 (Qwen2.5-Coder-7B, Llama-3.1-8B) × Tier 1 (100 ops × 5 lang + 50 code)

In [ ]:
!pip install -q transformers accelerate torch scipy matplotlib numpy

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title Stimuli Data (Tier 1: 50 comp + 50 judg)
import json

# Computational operations (50)
COMP_OPS = [
  {"id": "comp_01_sort_asc", "descriptions": {"en": "Sort the list in ascending order", "ko": "목록을 오름차순으로 정렬하라", "zh": "将列表按升序排序", "ar": "رتب القائمة بترتيب تصاعدي", "es": "Ordena la lista en orden ascendente"}},
  {"id": "comp_02_find_max", "descriptions": {"en": "Find the maximum value in the list", "ko": "목록에서 최댓값을 찾아라", "zh": "找到列表中的最大值", "ar": "أوجد القيمة القصوى في القائمة", "es": "Encuentra el valor máximo en la lista"}},
  {"id": "comp_03_filter_pos", "descriptions": {"en": "Filter only the positive numbers from the list", "ko": "목록에서 양수만 걸러내라", "zh": "从列表中筛选出正数", "ar": "صفّ الأرقام الموجبة فقط من القائمة", "es": "Filtra solo los números positivos de la lista"}},
  {"id": "comp_04_reverse", "descriptions": {"en": "Reverse the order of elements in the list", "ko": "목록의 원소 순서를 뒤집어라", "zh": "反转列表中元素的顺序", "ar": "اعكس ترتيب العناصر في القائمة", "es": "Invierte el orden de los elementos de la lista"}},
  {"id": "comp_05_count", "descriptions": {"en": "Count the number of elements in the list", "ko": "목록의 원소 개수를 세라", "zh": "计算列表中元素的数量", "ar": "احسب عدد العناصر في القائمة", "es": "Cuenta el número de elementos en la lista"}},
  {"id": "comp_06_sum", "descriptions": {"en": "Calculate the sum of all numbers in the list", "ko": "목록에 있는 모든 숫자의 합을 구하라", "zh": "计算列表中所有数字的总和", "ar": "احسب مجموع جميع الأرقام في القائمة", "es": "Calcula la suma de todos los números de la lista"}},
  {"id": "comp_07_deduplicate", "descriptions": {"en": "Remove duplicate elements from the list", "ko": "목록에서 중복된 원소를 제거하라", "zh": "从列表中去除重复元素", "ar": "أزل العناصر المكررة من القائمة", "es": "Elimina los elementos duplicados de la lista"}},
  {"id": "comp_08_top3", "descriptions": {"en": "Find the three largest values in the given list", "ko": "주어진 목록에서 가장 큰 값 세 개를 찾아라", "zh": "找到给定列表中最大的三个值", "ar": "أوجد أكبر ثلاث قيم في القائمة المعطاة", "es": "Encuentra los tres valores más grandes en la lista dada"}},
  {"id": "comp_09_mean", "descriptions": {"en": "Calculate the average of all numbers in the list", "ko": "목록에 있는 모든 숫자의 평균을 구하라", "zh": "计算列表中所有数字的平均值", "ar": "احسب متوسط جميع الأرقام في القائمة", "es": "Calcula el promedio de todos los números de la lista"}},
  {"id": "comp_10_sort_desc", "descriptions": {"en": "Sort the list in descending order", "ko": "목록을 내림차순으로 정렬하라", "zh": "将列表按降序排序", "ar": "رتب القائمة بترتيب تنازلي", "es": "Ordena la lista en orden descendente"}},
  {"id": "comp_11_concat", "descriptions": {"en": "Concatenate the two strings", "ko": "두 문자열을 이어붙여라", "zh": "连接两个字符串", "ar": "اربط السلسلتين النصيتين", "es": "Concatena las dos cadenas de texto"}},
  {"id": "comp_12_uppercase", "descriptions": {"en": "Convert the string to uppercase", "ko": "문자열을 대문자로 변환하라", "zh": "将字符串转换为大写", "ar": "حوّل النص إلى أحرف كبيرة", "es": "Convierte la cadena a mayúsculas"}},
  {"id": "comp_13_split", "descriptions": {"en": "Split the string by spaces into a list of words", "ko": "문자열을 공백 기준으로 나누어 단어 목록을 만들어라", "zh": "按空格将字符串拆分为单词列表", "ar": "قسّم النص بالمسافات إلى قائمة كلمات", "es": "Divide la cadena por espacios en una lista de palabras"}},
  {"id": "comp_14_replace", "descriptions": {"en": "Replace all occurrences of a with b in the string", "ko": "문자열에서 a를 모두 b로 바꿔라", "zh": "将字符串中所有的a替换为b", "ar": "استبدل جميع حالات a بـ b في النص", "es": "Reemplaza todas las apariciones de a por b en la cadena"}},
  {"id": "comp_15_length", "descriptions": {"en": "Calculate the length of the string", "ko": "문자열의 길이를 구하라", "zh": "计算字符串的长度", "ar": "احسب طول النص", "es": "Calcula la longitud de la cadena"}},
  {"id": "comp_16_abs", "descriptions": {"en": "Compute the absolute value of the number", "ko": "숫자의 절댓값을 구하라", "zh": "计算数字的绝对值", "ar": "احسب القيمة المطلقة للعدد", "es": "Calcula el valor absoluto del número"}},
  {"id": "comp_17_power", "descriptions": {"en": "Raise the number to the power of three", "ko": "숫자를 세제곱하라", "zh": "将数字求三次方", "ar": "ارفع العدد إلى القوة الثالثة", "es": "Eleva el número a la potencia de tres"}},
  {"id": "comp_18_modulo", "descriptions": {"en": "Compute the remainder when dividing by seven", "ko": "7로 나눈 나머지를 구하라", "zh": "计算除以七的余数", "ar": "احسب باقي القسمة على سبعة", "es": "Calcula el residuo de dividir entre siete"}},
  {"id": "comp_19_sqrt", "descriptions": {"en": "Calculate the square root of the number", "ko": "숫자의 제곱근을 구하라", "zh": "计算数字的平方根", "ar": "احسب الجذر التربيعي للعدد", "es": "Calcula la raíz cuadrada del número"}},
  {"id": "comp_20_gcd", "descriptions": {"en": "Find the greatest common divisor of the two numbers", "ko": "두 수의 최대공약수를 구하라", "zh": "求两个数的最大公约数", "ar": "أوجد القاسم المشترك الأكبر للعددين", "es": "Encuentra el máximo común divisor de los dos números"}},
  {"id": "comp_21_union", "descriptions": {"en": "Compute the union of the two sets", "ko": "두 집합의 합집합을 구하라", "zh": "计算两个集合的并集", "ar": "احسب اتحاد المجموعتين", "es": "Calcula la unión de los dos conjuntos"}},
  {"id": "comp_22_intersect", "descriptions": {"en": "Find the common elements between the two sets", "ko": "두 집합의 공통 원소를 찾아라", "zh": "找出两个集合的公共元素", "ar": "أوجد العناصر المشتركة بين المجموعتين", "es": "Encuentra los elementos comunes entre los dos conjuntos"}},
  {"id": "comp_23_difference", "descriptions": {"en": "Find elements in the first set that are not in the second", "ko": "첫 번째 집합에만 있는 원소를 찾아라", "zh": "找出在第一个集合中但不在第二个集合中的元素", "ar": "أوجد العناصر الموجودة في المجموعة الأولى وليست في الثانية", "es": "Encuentra los elementos del primer conjunto que no están en el segundo"}},
  {"id": "comp_24_keys", "descriptions": {"en": "Extract all keys from the dictionary", "ko": "사전에서 모든 키를 추출하라", "zh": "从字典中提取所有键", "ar": "استخرج جميع المفاتيح من القاموس", "es": "Extrae todas las claves del diccionario"}},
  {"id": "comp_25_merge", "descriptions": {"en": "Merge the two dictionaries into one", "ko": "두 사전을 하나로 합쳐라", "zh": "将两个字典合并为一个", "ar": "ادمج القاموسين في واحد", "es": "Fusiona los dos diccionarios en uno"}},
  {"id": "comp_26_transpose", "descriptions": {"en": "Transpose the matrix", "ko": "행렬을 전치하라", "zh": "转置矩阵", "ar": "انقل المصفوفة", "es": "Transpón la matriz"}},
  {"id": "comp_27_flatten", "descriptions": {"en": "Flatten a nested list into a single list", "ko": "중첩된 목록을 단일 목록으로 펼쳐라", "zh": "将嵌套列表展平为单个列表", "ar": "افرد القائمة المتداخلة إلى قائمة واحدة", "es": "Aplana una lista anidada en una sola lista"}},
  {"id": "comp_28_zip", "descriptions": {"en": "Pair up elements from two lists by position", "ko": "두 목록의 원소를 위치별로 짝지어라", "zh": "按位置将两个列表的元素配对", "ar": "قرن عناصر القائمتين حسب الموضع", "es": "Empareja elementos de dos listas por posición"}},
  {"id": "comp_29_index", "descriptions": {"en": "Find the position of a target value in the list", "ko": "목록에서 목표 값의 위치를 찾아라", "zh": "找到目标值在列表中的位置", "ar": "أوجد موضع القيمة المستهدفة في القائمة", "es": "Encuentra la posición de un valor objetivo en la lista"}},
  {"id": "comp_30_contains", "descriptions": {"en": "Check whether the list contains the target value", "ko": "목록에 목표 값이 있는지 확인하라", "zh": "检查列表是否包含目标值", "ar": "تحقق مما إذا كانت القائمة تحتوي على القيمة المستهدفة", "es": "Verifica si la lista contiene el valor objetivo"}},
  {"id": "comp_31_all_pos", "descriptions": {"en": "Check if all numbers in the list are positive", "ko": "목록의 모든 숫자가 양수인지 확인하라", "zh": "检查列表中的所有数字是否为正数", "ar": "تحقق مما إذا كانت جميع الأرقام في القائمة موجبة", "es": "Verifica si todos los números de la lista son positivos"}},
  {"id": "comp_32_any_neg", "descriptions": {"en": "Check if any number in the list is negative", "ko": "목록에 음수가 하나라도 있는지 확인하라", "zh": "检查列表中是否有负数", "ar": "تحقق مما إذا كان أي رقم في القائمة سالبًا", "es": "Verifica si algún número de la lista es negativo"}},
  {"id": "comp_33_int_to_str", "descriptions": {"en": "Convert the integer to a string", "ko": "정수를 문자열로 변환하라", "zh": "将整数转换为字符串", "ar": "حوّل العدد الصحيح إلى نص", "es": "Convierte el entero a una cadena"}},
  {"id": "comp_34_round", "descriptions": {"en": "Round the number to two decimal places", "ko": "숫자를 소수점 둘째 자리로 반올림하라", "zh": "将数字四舍五入到小数点后两位", "ar": "قرّب العدد إلى منزلتين عشريتين", "es": "Redondea el número a dos decimales"}},
  {"id": "comp_35_min", "descriptions": {"en": "Find the minimum value in the list", "ko": "목록에서 최솟값을 찾아라", "zh": "找到列表中的最小值", "ar": "أوجد القيمة الصغرى في القائمة", "es": "Encuentra el valor mínimo en la lista"}},
  {"id": "comp_36_median", "descriptions": {"en": "Find the median value of the list", "ko": "목록의 중앙값을 구하라", "zh": "找到列表的中位数", "ar": "أوجد القيمة الوسطى للقائمة", "es": "Encuentra el valor mediano de la lista"}},
  {"id": "comp_37_slice", "descriptions": {"en": "Extract elements from index 2 to 4", "ko": "인덱스 2부터 4까지의 원소를 추출하라", "zh": "提取索引2到4的元素", "ar": "استخرج العناصر من الفهرس 2 إلى 4", "es": "Extrae los elementos del índice 2 al 4"}},
  {"id": "comp_38_even", "descriptions": {"en": "Filter only the even numbers from the list", "ko": "목록에서 짝수만 걸러내라", "zh": "从列表中筛选出偶数", "ar": "صفّ الأرقام الزوجية فقط من القائمة", "es": "Filtra solo los números pares de la lista"}},
  {"id": "comp_39_freq", "descriptions": {"en": "Count the frequency of each element in the list", "ko": "목록에서 각 원소의 빈도를 세라", "zh": "计算列表中每个元素的频率", "ar": "احسب تكرار كل عنصر في القائمة", "es": "Cuenta la frecuencia de cada elemento en la lista"}},
  {"id": "comp_40_map_double", "descriptions": {"en": "Double every element in the list", "ko": "목록의 모든 원소를 두 배로 만들어라", "zh": "将列表中的每个元素翻倍", "ar": "ضاعف كل عنصر في القائمة", "es": "Duplica cada elemento de la lista"}},
  {"id": "comp_41_depth", "descriptions": {"en": "Calculate the depth of a tree", "ko": "트리의 깊이를 계산하라", "zh": "计算树的深度", "ar": "احسب عمق الشجرة", "es": "Calcula la profundidad de un árbol"}},
  {"id": "comp_42_is_palindrome", "descriptions": {"en": "Check whether the string is a palindrome", "ko": "문자열이 회문인지 확인하라", "zh": "检查字符串是否为回文", "ar": "تحقق مما إذا كان النص متناظرًا", "es": "Verifica si la cadena es un palíndromo"}},
  {"id": "comp_43_binary", "descriptions": {"en": "Convert the number to binary representation", "ko": "숫자를 이진 표현으로 변환하라", "zh": "将数字转换为二进制表示", "ar": "حوّل العدد إلى تمثيل ثنائي", "es": "Convierte el número a representación binaria"}},
  {"id": "comp_44_cumsum", "descriptions": {"en": "Compute the cumulative sum of the list", "ko": "목록의 누적 합을 구하라", "zh": "计算列表的累积和", "ar": "احسب المجموع التراكمي للقائمة", "es": "Calcula la suma acumulada de la lista"}},
  {"id": "comp_45_prod", "descriptions": {"en": "Calculate the product of all numbers in the list", "ko": "목록에 있는 모든 숫자의 곱을 구하라", "zh": "计算列表中所有数字的乘积", "ar": "احسب حاصل ضرب جميع الأرقام في القائمة", "es": "Calcula el producto de todos los números de la lista"}},
  {"id": "comp_46_range", "descriptions": {"en": "Generate a list of numbers from 1 to 10", "ko": "1부터 10까지의 숫자 목록을 생성하라", "zh": "生成从1到10的数字列表", "ar": "أنشئ قائمة أرقام من 1 إلى 10", "es": "Genera una lista de números del 1 al 10"}},
  {"id": "comp_47_char_count", "descriptions": {"en": "Count the number of characters in the string", "ko": "문자열의 문자 수를 세라", "zh": "计算字符串中的字符数", "ar": "احسب عدد الأحرف في النص", "es": "Cuenta el número de caracteres en la cadena"}},
  {"id": "comp_48_prime", "descriptions": {"en": "Check whether the number is prime", "ko": "숫자가 소수인지 확인하라", "zh": "检查数字是否为素数", "ar": "تحقق مما إذا كان العدد أوليًا", "es": "Verifica si el número es primo"}},
  {"id": "comp_49_sort_by_len", "descriptions": {"en": "Sort the list of strings by their length", "ko": "문자열 목록을 길이순으로 정렬하라", "zh": "按长度对字符串列表排序", "ar": "رتب قائمة النصوص حسب طولها", "es": "Ordena la lista de cadenas por su longitud"}},
  {"id": "comp_50_matrix_mult", "descriptions": {"en": "Multiply two matrices", "ko": "두 행렬을 곱하라", "zh": "将两个矩阵相乘", "ar": "اضرب مصفوفتين", "es": "Multiplica dos matrices"}}
]

# Judgment operations (first 10 as representative sample)
JUDG_OPS = [
  {"id": "judg_01_prioritize", "descriptions": {"en": "Prioritize the tasks by importance", "ko": "중요도에 따라 작업의 우선순위를 정하라", "zh": "按重要性排列任务的优先级", "ar": "رتب المهام حسب الأهمية", "es": "Prioriza las tareas por importancia"}},
  {"id": "judg_02_evaluate", "descriptions": {"en": "Evaluate the quality of the work", "ko": "작업의 품질을 평가하라", "zh": "评估工作的质量", "ar": "قيّم جودة العمل", "es": "Evalúa la calidad del trabajo"}},
  {"id": "judg_03_recommend", "descriptions": {"en": "Recommend the best option from the list", "ko": "목록에서 최선의 선택을 추천하라", "zh": "从列表中推荐最佳选项", "ar": "أوصِ بأفضل خيار من القائمة", "es": "Recomienda la mejor opción de la lista"}},
  {"id": "judg_04_classify", "descriptions": {"en": "Classify the items into categories", "ko": "항목들을 범주로 분류하라", "zh": "将项目分类", "ar": "صنّف العناصر إلى فئات", "es": "Clasifica los elementos en categorías"}},
  {"id": "judg_05_compare", "descriptions": {"en": "Compare the two options and choose the better one", "ko": "두 선택지를 비교하여 더 나은 것을 고르라", "zh": "比较两个选项并选择更好的一个", "ar": "قارن بين الخيارين واختر الأفضل", "es": "Compara las dos opciones y elige la mejor"}},
  {"id": "judg_06_summarize", "descriptions": {"en": "Summarize the key points", "ko": "핵심 요점을 요약하라", "zh": "总结要点", "ar": "لخّص النقاط الرئيسية", "es": "Resume los puntos clave"}},
  {"id": "judg_07_rank", "descriptions": {"en": "Rank the candidates by suitability", "ko": "적합도에 따라 후보를 순위 매기라", "zh": "按适合度对候选人排名", "ar": "رتب المرشحين حسب الملاءمة", "es": "Clasifica a los candidatos por idoneidad"}},
  {"id": "judg_08_decide", "descriptions": {"en": "Decide whether to approve or reject", "ko": "승인할지 거부할지 결정하라", "zh": "决定是批准还是拒绝", "ar": "قرر ما إذا كنت ستوافق أو ترفض", "es": "Decide si aprobar o rechazar"}},
  {"id": "judg_09_assess_risk", "descriptions": {"en": "Assess the risk level of the situation", "ko": "상황의 위험 수준을 평가하라", "zh": "评估情况的风险等级", "ar": "قيّم مستوى خطورة الموقف", "es": "Evalúa el nivel de riesgo de la situación"}},
  {"id": "judg_10_interpret", "descriptions": {"en": "Interpret the meaning of the results", "ko": "결과의 의미를 해석하라", "zh": "解释结果的含义", "ar": "فسّر معنى النتائج", "es": "Interpreta el significado de los resultados"}}
]

CODE_EQUIVALENTS = {
  "comp_01_sort_asc": "sorted(lst)", "comp_02_find_max": "max(lst)",
  "comp_03_filter_pos": "[x for x in lst if x > 0]", "comp_04_reverse": "lst[::-1]",
  "comp_05_count": "len(lst)", "comp_06_sum": "sum(lst)",
  "comp_07_deduplicate": "list(set(lst))", "comp_08_top3": "sorted(lst, reverse=True)[:3]",
  "comp_09_mean": "sum(lst) / len(lst)", "comp_10_sort_desc": "sorted(lst, reverse=True)",
  "comp_11_concat": "s1 + s2", "comp_12_uppercase": "s.upper()",
  "comp_13_split": "s.split()", "comp_14_replace": "s.replace('a', 'b')",
  "comp_15_length": "len(s)", "comp_16_abs": "abs(n)",
  "comp_17_power": "n ** 3", "comp_18_modulo": "n % 7",
  "comp_19_sqrt": "n ** 0.5", "comp_20_gcd": "math.gcd(a, b)",
  "comp_21_union": "s1 | s2", "comp_22_intersect": "s1 & s2",
  "comp_23_difference": "s1 - s2", "comp_24_keys": "list(d.keys())",
  "comp_25_merge": "{**d1, **d2}", "comp_26_transpose": "list(zip(*matrix))",
  "comp_27_flatten": "[x for sub in lst for x in sub]",
  "comp_28_zip": "list(zip(lst1, lst2))", "comp_29_index": "lst.index(target)",
  "comp_30_contains": "target in lst", "comp_31_all_pos": "all(x > 0 for x in lst)",
  "comp_32_any_neg": "any(x < 0 for x in lst)", "comp_33_int_to_str": "str(n)",
  "comp_34_round": "round(n, 2)", "comp_35_min": "min(lst)",
  "comp_36_median": "sorted(lst)[len(lst)//2]", "comp_37_slice": "lst[2:5]",
  "comp_38_even": "[x for x in lst if x % 2 == 0]",
  "comp_39_freq": "collections.Counter(lst)",
  "comp_40_map_double": "[x * 2 for x in lst]",
  "comp_41_depth": "def depth(t): return 1 + max(depth(c) for c in t.children) if t.children else 0",
  "comp_42_is_palindrome": "s == s[::-1]", "comp_43_binary": "bin(n)",
  "comp_44_cumsum": "[sum(lst[:i+1]) for i in range(len(lst))]",
  "comp_45_prod": "functools.reduce(lambda a, b: a * b, lst)",
  "comp_46_range": "list(range(1, 11))", "comp_47_char_count": "len(s)",
  "comp_48_prime": "all(n % i != 0 for i in range(2, int(n**0.5)+1)) and n > 1",
  "comp_49_sort_by_len": "sorted(lst, key=len)",
  "comp_50_matrix_mult": "[[sum(a*b for a,b in zip(row,col)) for col in zip(*B)] for row in A]"
}

LANGUAGES = ["en", "ko", "zh", "ar", "es"]
print(f"Loaded {len(COMP_OPS)} comp + {len(JUDG_OPS)} judg ops, {len(CODE_EQUIVALENTS)} code snippets")

In [ ]:
#@title Core extraction + analysis functions
import numpy as np
import time
from itertools import combinations
from scipy.spatial.distance import cosine
from transformers import AutoModelForCausalLM, AutoTokenizer

def run_model(model_name, comp_ops, judg_ops, code_equiv, languages):
    """Full pipeline: load model → extract → analyze → plot."""
    print(f"\n{'='*60}")
    print(f"MODEL: {model_name}")
    print(f"{'='*60}")

    # Load
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.float16,
        output_hidden_states=True, trust_remote_code=True
    ).cuda()
    model.eval()
    n_layers = model.config.num_hidden_layers
    hidden_dim = model.config.hidden_size
    print(f"Loaded in {time.time()-t0:.0f}s: {n_layers} layers, {hidden_dim}d")

    def extract(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            out = model(**inputs)
        return np.stack([h[0, -1, :].cpu().float().numpy() for h in out.hidden_states])

    # Extract NL
    all_ops = comp_ops + judg_ops
    nl_states = {}
    total = len(all_ops) * len(languages)
    for i, op in enumerate(all_ops):
        nl_states[op["id"]] = {}
        for lang in languages:
            desc = op.get("descriptions", {}).get(lang)
            if desc:
                nl_states[op["id"]][lang] = extract(desc)
        if (i+1) % 20 == 0:
            print(f"  NL: {(i+1)*len(languages)}/{total}")
    print(f"  NL done: {len(nl_states)} ops")

    # Extract code
    code_states = {}
    for op in comp_ops:
        code = code_equiv.get(op["id"])
        if code:
            code_states[op["id"]] = extract(code)
    print(f"  Code done: {len(code_states)} snippets")

    # Free GPU
    del model
    torch.cuda.empty_cache()

    # Analyze
    comp_ids = [op["id"] for op in comp_ops]
    judg_ids = [op["id"] for op in judg_ops]
    R_all = np.zeros(n_layers + 1)
    R_C = np.zeros(n_layers + 1)
    R_J = np.zeros(n_layers + 1)
    R_code = np.zeros(n_layers + 1)

    for layer in range(n_layers + 1):
        # R_all
        d_intra, d_inter = [], []
        for op_id in nl_states:
            vecs = [nl_states[op_id][l][layer] for l in languages if l in nl_states[op_id]]
            for a, b in combinations(vecs, 2): d_intra.append(cosine(a, b))
        for lang in languages:
            vecs = [nl_states[op_id][lang][layer] for op_id in nl_states if lang in nl_states[op_id]]
            for a, b in combinations(vecs, 2): d_inter.append(cosine(a, b))
        R_all[layer] = np.mean(d_inter) / max(np.mean(d_intra), 1e-10)

        # R_C
        di, de = [], []
        for op_id in comp_ids:
            vecs = [nl_states[op_id][l][layer] for l in languages if l in nl_states.get(op_id, {})]
            for a, b in combinations(vecs, 2): di.append(cosine(a, b))
        for lang in languages:
            vecs = [nl_states[op_id][lang][layer] for op_id in comp_ids if lang in nl_states.get(op_id, {})]
            for a, b in combinations(vecs, 2): de.append(cosine(a, b))
        R_C[layer] = np.mean(de) / max(np.mean(di), 1e-10) if di else 0

        # R_J
        di, de = [], []
        for op_id in judg_ids:
            vecs = [nl_states[op_id][l][layer] for l in languages if l in nl_states.get(op_id, {})]
            for a, b in combinations(vecs, 2): di.append(cosine(a, b))
        for lang in languages:
            vecs = [nl_states[op_id][lang][layer] for op_id in judg_ids if lang in nl_states.get(op_id, {})]
            for a, b in combinations(vecs, 2): de.append(cosine(a, b))
        R_J[layer] = np.mean(de) / max(np.mean(di), 1e-10) if di else 0

        # R_code
        dm, dmm = [], []
        for op_id in comp_ids:
            if op_id not in code_states: continue
            cv = code_states[op_id][layer]
            for lang in languages:
                if lang not in nl_states.get(op_id, {}): continue
                nv = nl_states[op_id][lang][layer]
                dm.append(cosine(nv, cv))
                for oid in comp_ids[:10]:
                    if oid != op_id and oid in code_states:
                        dmm.append(cosine(nv, code_states[oid][layer]))
        R_code[layer] = np.mean(dmm) / max(np.mean(dm), 1e-10) if dm else 0

    safe = model_name.split('/')[-1]
    results = {
        "model": model_name, "n_layers": n_layers, "hidden_dim": hidden_dim,
        "R_all": R_all.tolist(), "R_C": R_C.tolist(), "R_J": R_J.tolist(),
        "R_code": R_code.tolist(),
        "peak_R_all": {"layer": int(np.argmax(R_all)), "value": float(np.max(R_all))},
        "peak_R_code": {"layer": int(np.argmax(R_code)), "value": float(np.max(R_code))},
        "p2_holds_layers": [int(l) for l in range(n_layers+1) if R_C[l] > R_J[l]],
    }

    print(f"\nPeak R: layer {results['peak_R_all']['layer']}, R={results['peak_R_all']['value']:.3f}")
    print(f"Peak R_code: layer {results['peak_R_code']['layer']}, R_code={results['peak_R_code']['value']:.3f}")
    print(f"P2 holds at: {results['p2_holds_layers'] or 'NONE'}")

    return results, R_all, R_C, R_J, R_code

In [ ]:
#@title Run Model 1: Qwen2.5-Coder-7B-Instruct
r1, R1_all, R1_C, R1_J, R1_code = run_model(
    "Qwen/Qwen2.5-Coder-7B-Instruct", COMP_OPS, JUDG_OPS, CODE_EQUIVALENTS, LANGUAGES)

In [ ]:
#@title Run Model 2: Llama-3.1-8B-Instruct
r2, R2_all, R2_C, R2_J, R2_code = run_model(
    "meta-llama/Llama-3.1-8B-Instruct", COMP_OPS, JUDG_OPS, CODE_EQUIVALENTS, LANGUAGES)

In [ ]:
#@title Plot Results
import matplotlib.pyplot as plt

models = [(r1, R1_all, R1_C, R1_J, R1_code), (r2, R2_all, R2_C, R2_J, R2_code)]

fig, axes = plt.subplots(len(models), 3, figsize=(18, 5*len(models)))
if len(models) == 1: axes = [axes]

for i, (res, Ra, Rc, Rj, Rcd) in enumerate(models):
    safe = res['model'].split('/')[-1]
    layers = np.arange(len(Ra))

    # Convergence curve
    axes[i][0].plot(layers, Ra, 'o-', color='#1f77b4', lw=2, ms=3)
    axes[i][0].axhline(1.0, color='gray', ls='--', alpha=0.5)
    pk = np.argmax(Ra)
    axes[i][0].annotate(f'peak L{pk}\nR={Ra[pk]:.2f}', xy=(pk, Ra[pk]),
        xytext=(pk+2, Ra[pk]+0.05), arrowprops=dict(arrowstyle='->', color='red'),
        fontsize=9, color='red')
    axes[i][0].set_title(f'{safe}\nConvergence Curve')
    axes[i][0].set_xlabel('Layer'); axes[i][0].set_ylabel('R')

    # P2
    axes[i][1].plot(layers, Rc, 'o-', color='#2ca02c', lw=2, ms=3, label='R_C')
    axes[i][1].plot(layers, Rj, 's-', color='#d62728', lw=2, ms=3, label='R_J')
    axes[i][1].fill_between(layers, Rc, Rj, where=Rc>Rj, alpha=0.15, color='green')
    axes[i][1].fill_between(layers, Rc, Rj, where=Rc<=Rj, alpha=0.15, color='red')
    axes[i][1].axhline(1.0, color='gray', ls='--', alpha=0.5)
    axes[i][1].set_title(f'{safe}\nP2 (R_C vs R_J)'); axes[i][1].legend()
    axes[i][1].set_xlabel('Layer'); axes[i][1].set_ylabel('R')

    # Overlay
    axes[i][2].plot(layers, Ra, 'o-', color='#1f77b4', lw=2, ms=3, label='R (cross-ling)')
    axes[i][2].plot(layers, Rcd, 's-', color='#8c564b', lw=2, ms=3, label='R_code')
    axes[i][2].axhline(1.0, color='gray', ls='--', alpha=0.5)
    axes[i][2].set_title(f'{safe}\nCross-ling vs NL-Code'); axes[i][2].legend()
    axes[i][2].set_xlabel('Layer'); axes[i][2].set_ylabel('R')

fig.tight_layout()
fig.savefig('v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v2_results.png')

In [ ]:
#@title Save Results JSON
all_results = {"qwen_coder_7b": r1, "llama_8b": r2}
with open("v2_hidden_state_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved: v2_hidden_state_results.json")

# Download files
from google.colab import files
files.download("v2_results.png")
files.download("v2_hidden_state_results.json")